In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
dbutils.widgets.text("increm_flag","0")
increm_flag = dbutils.widgets.get("increm_flag")

In [0]:
df_src = spark.read.format("delta").load("abfss://flights@adls7786.dfs.core.windows.net/flights/bronze").drop('_rescued_data')
df_src.display()


In [0]:
if spark.catalog.tableExists("flights.silver.dim_flights"):
    df_sink = spark.sql("select * from flights.silver.dim_flights")
else:
    df_sink = spark.sql("select *,0 dim_flight_key,0 create_date,0 update_date from parquet.`abfss://flights@adls7786.dfs.core.windows.net/flights/bronze` where 1=0").drop('_rescued_data')
df_sink.display()

In [0]:
df_sink = df_sink.withColumnRenamed("dim_flight_key","old_dim_flight_key") \
            .withColumnRenamed("create_date","old_create_date") \
            .withColumnRenamed("update_date","old_update_date")
df_sink.display()


In [0]:
df_join = df_src.join(df_sink,df_src['flight_id'] == df_sink['flight_id'],'left').select(df_src['*'],df_sink['old_dim_flight_key'],df_sink['old_create_date'],df_sink['old_update_date'])
df_join.display()

In [0]:
df_old = df_join.where(df_join['old_dim_flight_key'].isNotNull())
df_new = df_join.where(df_join['old_dim_flight_key'].isNull())
df_new.display()
df_old.display()



In [0]:
df_old = df_old.drop("old_update_date")
df_old = df_old.withColumnRenamed("old_dim_flight_key","dim_flight_key")
df_old = df_old.withColumnRenamed("old_create_date",'create_date')
df_old = df_old.withColumn("create_date",F.to_timestamp(df_old['create_date']))
df_old = df_old.withColumn("update_date",F.current_timestamp())
df_old.display()


In [0]:
df_new = df_new.drop("old_dim_flight_key","old_create_date","old_update_date")
df_new = df_new.withColumn("dim_flight_key",F.monotonically_increasing_id()+ F.lit(1))
df_new = df_new.withColumn("create_date",F.current_timestamp())
df_new = df_new.withColumn("update_date",F.current_timestamp())
df_new.display()


In [0]:
if (increm_flag == "0"):
    max_value = 1
else:
    if spark.catalog.tableExists("flights.silver.dim_flights"):
        df = spark.sql("select max(dim_flight_key) from flights.silver.dim_flights ")
        max_value = df.collect()[0][0]
max_value

In [0]:
df_new = df_new.withColumn("dim_flight_key",F.lit(max_value) + F.monotonically_increasing_id())
df_new.display()

In [0]:
df_final = df_new.unionByName(df_old)
df_final.display()

In [0]:
if spark.catalog.tableExists("flights.silver.dim_flights"):
    delta_table = DeltaTable.forName(spark,"flights.silver.dim_flights")
    delta_table.alias('t').merge(df_final.alias('s'),"t.dim_flight_key = s.dim_flight_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("overwrite").option("path","abfss://flights@adls7786.dfs.core.windows.net/flights/silver/dim_flights").saveAsTable("flights.silver.dim_flights")

In [0]:
%sql

select * from flights.silver.dim_flights;